# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeroTechno/Machine-Learning/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We engineer a clean feature matrix combining historical search performance metrics (avg_position, impressions_90d, ctr) with page structural metadata (word_count, days_since_last_update). Missing values are safely imputed with zero.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url).fillna(0)

# Create Target Proxy
df["target"] = ((df["impressions_90d"] >= 100) & (df["ctr"] >= 0.05)).astype(int)

# Build Feature Vector X
feature_cols = ["avg_position", "impressions_90d", "ctr", "word_count", "days_since_last_update"]
X = df[feature_cols].copy()
y = df["target"]

print(f"Feature Vector Built: {X.shape[0]} rows x {X.shape[1]} features")
print(X.head(3))

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature Metadata Breakdown:

avg_position: Mean SERP ranking position. Nulls filled with 0. Knowable prior to decision from Search Console history.

impressions_90d: Total search visibility over past 90 days. Nulls filled with 0. Knowable from logged search traffic.

ctr: Historical click-through rate. Nulls filled with 0. Knowable prior to decision day.

word_count: Article word length. Nulls filled with 0. Knowable by scraping the current published HTML.

days_since_last_update: Content age in days. Nulls filled with 0. Knowable from CMS publication timestamps.

In [ ]:
feature_notes = pd.DataFrame({
    "Feature": feature_cols,
    "Missing_Handling": ["fillna(0)"] * 5,
    "Available_Before_Decision": ["YES"] * 5
})

print(feature_notes.to_string(index=False))

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage Experiment: We intentionally inject a synthetic leaky feature derived directly from the target label (LEAKY_future_clicks), observe the model score jump to an artificial 100% accuracy, and then purge the leaky column to restore an honest baseline model.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# 1. Honest Model
honest_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X, y)
print(f"Honest Model Score: {honest_tree.score(X, y):.4f}")

# 2. Inject Leaky Feature
X_leaky = X.copy()
X_leaky["LEAKY_future_clicks"] = y * 1000 + np.random.normal(0, 1, len(y))
leaky_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
print(f"Leaky Model Score:  {leaky_tree.score(X_leaky, y):.4f} <-- Artificial Perfect Score!")

# 3. Delete Leaky Feature
X_cleaned = X_leaky.drop(columns=["LEAKY_future_clicks"])
print(f"Cleaned Columns: {list(X_cleaned.columns)}")

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

future_clicks: Excluded because future post-decision click volume is unknown at prediction time and creates severe temporal data leakage.

future_impressions: Excluded to prevent predicting outcomes using future post-decision traffic metrics.

page_url: Excluded to prevent the model from memorizing specific URL paths or brand strings rather than learning generalizable signals.

In [ ]:
excluded_fields = {
    "future_clicks": "Temporal leakage (post-decision outcome)",
    "future_impressions": "Temporal leakage (post-decision outcome)",
    "page_url": "High cardinality string; causes memorization"
}

for field, reason in excluded_fields.items():
    print(f"Excluded '{field}': {reason}")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.